# This Notebook Serves as main stream of task3

- Group machines and parts in product families using DCA to split overall capabiities in each product family `center`
- use functional organization within each `department` and draw from-to-chart for each functional `cell`
- draw detailed layout within each functional cell

## DCA Machine Grouping Analysis
This notebook creates an Step-product incidence matrix and prepares it for Direct Clustering Algorithm (DCA) analysis.

In [88]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import math
from collections import defaultdict
import json

In [89]:
# Load the data
steps_df = pd.read_csv('../Data/Steps.csv')
equipment_df = pd.read_csv('../Data/Equipment.csv')

In [90]:
print(steps_df)

   Part Step 1 Step 2 Step 3 Step 4 Step 5 Step 6 Step 7
0    P1      B      A      B      C      D      I      J
1    P2      A      C      D      H      J    NaN    NaN
2    P3      B      D      C      I      J    NaN    NaN
3    P4      A      B      D      G      H    NaN    NaN
4    P5      B      C      D      I    NaN    NaN    NaN
5    P6      A      B      C      D      H      I      J
6    P7      E      F      C      D      I      J    NaN
7    P8      E      H      J      I    NaN    NaN    NaN
8    P9      F      G      E      G      I      J    NaN
9   P10      E      F      I      J    NaN    NaN    NaN
10  P11      E      G      E      G      I    NaN    NaN
11  P12      E      G      F      I      J    NaN    NaN
12  P13      E      F      G      F      G      H      I
13  P14      E      F      G      H    NaN    NaN    NaN
14  P15      E      G      F      H      J    NaN    NaN
15  P16      F      H      I      J    NaN    NaN    NaN
16  P17      K      L      M   

In [91]:
# Parse the equipment names

equipment_list = equipment_df['Equipment'].tolist()
products = steps_df['Part'].tolist()

In [ ]:
# Get all unique steps required across all products
all_steps = set()
for idx, row in steps_df.iterrows():
    for col in steps_df.columns[1:]:  # Skip the 'Part' column
        step = row[col]
        if pd.notna(step) and step != '':
            all_steps.add(step)

all_steps = sorted(list(all_steps))
print(f"Unique steps: {all_steps}")

Unique steps: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M']


In [93]:
# dictionary mapping each product to the set of steps it requires
product_steps = {}
for idx, row in steps_df.iterrows():
    product = row['Part']
    steps_required = set()
    for col in steps_df.columns[1:]:  # Skip the 'Part' column
        step = row[col]
        if pd.notna(step) and step != '':
            steps_required.add(step)
    product_steps[product] = steps_required


# # dictionary mapping each equipment into steps it can be done
# equipment_capabilities = {}
# for idx, row in equipment_df.iterrows():
#     equipment = row['Equipment']
#     caps = set()
#     caps = set(ch for ch in str(equipment) if ch.isalpha() and ch.isupper())
#     equipment_capabilities[equipment] = caps

print(product_steps)

{'P1': {'I', 'B', 'D', 'J', 'C', 'A'}, 'P2': {'D', 'J', 'C', 'A', 'H'}, 'P3': {'I', 'B', 'D', 'J', 'C'}, 'P4': {'G', 'B', 'D', 'A', 'H'}, 'P5': {'I', 'D', 'C', 'B'}, 'P6': {'I', 'B', 'D', 'J', 'C', 'A', 'H'}, 'P7': {'I', 'E', 'D', 'J', 'C', 'F'}, 'P8': {'I', 'J', 'E', 'H'}, 'P9': {'I', 'E', 'G', 'J', 'F'}, 'P10': {'I', 'J', 'E', 'F'}, 'P11': {'I', 'E', 'G'}, 'P12': {'I', 'E', 'G', 'J', 'F'}, 'P13': {'I', 'E', 'G', 'F', 'H'}, 'P14': {'E', 'F', 'G', 'H'}, 'P15': {'E', 'G', 'J', 'F', 'H'}, 'P16': {'I', 'J', 'F', 'H'}, 'P17': {'M', 'K', 'L'}, 'P18': {'M', 'K', 'L'}, 'P19': {'M', 'L'}, 'P20': {'M', 'K', 'L'}}


In [94]:
# create the relationship matrix 
matrix = np.zeros((len(product_steps), len(all_steps)), dtype=int)
# This creates (20 products × 13 steps) matrix ✓

for i, (part, req) in enumerate(product_steps.items()):  
    for j, step in enumerate(all_steps):                 
        if step in req:
            matrix[i, j] = 1 

incidence_df = pd.DataFrame(matrix, index=products, columns=all_steps)
# transposed_matrix = incidence_df.T

print(incidence_df)

     A  B  C  D  E  F  G  H  I  J  K  L  M
P1   1  1  1  1  0  0  0  0  1  1  0  0  0
P2   1  0  1  1  0  0  0  1  0  1  0  0  0
P3   0  1  1  1  0  0  0  0  1  1  0  0  0
P4   1  1  0  1  0  0  1  1  0  0  0  0  0
P5   0  1  1  1  0  0  0  0  1  0  0  0  0
P6   1  1  1  1  0  0  0  1  1  1  0  0  0
P7   0  0  1  1  1  1  0  0  1  1  0  0  0
P8   0  0  0  0  1  0  0  1  1  1  0  0  0
P9   0  0  0  0  1  1  1  0  1  1  0  0  0
P10  0  0  0  0  1  1  0  0  1  1  0  0  0
P11  0  0  0  0  1  0  1  0  1  0  0  0  0
P12  0  0  0  0  1  1  1  0  1  1  0  0  0
P13  0  0  0  0  1  1  1  1  1  0  0  0  0
P14  0  0  0  0  1  1  1  1  0  0  0  0  0
P15  0  0  0  0  1  1  1  1  0  1  0  0  0
P16  0  0  0  0  0  1  0  1  1  1  0  0  0
P17  0  0  0  0  0  0  0  0  0  0  1  1  1
P18  0  0  0  0  0  0  0  0  0  0  1  1  1
P19  0  0  0  0  0  0  0  0  0  0  0  1  1
P20  0  0  0  0  0  0  0  0  0  0  1  1  1


### DCA Algorithm Implementation

In [95]:
def dca_step1_initial_ordering(matrix):
    """
    Order rows and columns by sum of 1s
    """
    row_sums = matrix.sum(axis=1)
    col_sums = matrix.sum(axis=0)
    
    # Sort rows in DESCENDING order by sum
    # For ties, use descending order of original index
    row_order = row_sums.sort_values(ascending=False, kind='stable')
    sorted_rows = matrix.loc[row_order.index]
    
    col_order = col_sums.sort_values(ascending=True, kind='stable')
    sorted_matrix = sorted_rows[col_order.index]
    
    return sorted_matrix


def dca_step2_shift_columns_left(matrix):
    """
    Sort columns by shifting columns with 1s to the left
    - Starting with first row, shift all columns with 1 in that row to the left
    - Continue row by row until no further shifts possible
    """
    result = matrix.copy()
    cols = list(result.columns)
    
    shifted_cols = []
    remaining_cols = cols.copy()
    
    for row_idx in result.index:
        row = result.loc[row_idx]
        
        # Find columns with 1s in this row that haven't been shifted yet
        cols_with_ones = [col for col in remaining_cols if row[col] == 1]
        
        # Add these columns to the shifted list
        shifted_cols.extend(cols_with_ones)
        
        for col in cols_with_ones:
            remaining_cols.remove(col)
    
    # Add any remaining columns at the end
    shifted_cols.extend(remaining_cols)
    
    # Reorder the matrix with new column order
    result = result[shifted_cols]
    
    # print("\nStep 2: Column Left-Shifting")
    # print(f"New column order: {shifted_cols}")
    
    return result


def dca_step3_shift_rows_up(matrix):
    """
    Sort rows by shifting rows with 1s upward
    """
    result = matrix.copy()
    rows = list(result.index)
    
    shifted_rows = []
    remaining_rows = rows.copy()
    
    for col_name in result.columns:
        col = result[col_name]

        rows_with_ones = [row for row in remaining_rows if col[row] == 1]
        
        shifted_rows.extend(rows_with_ones)
        
        for row in rows_with_ones:
            remaining_rows.remove(row)
    
    shifted_rows.extend(remaining_rows)
    result = result.loc[shifted_rows]
    
    # print("\nStep 3: Row Upward-Shifting")
    # print(f"New row order: {shifted_rows}")
    
    return result


def dca_algorithm(matrix):

    print("="*80)
    print("DCA ALGORITHM - Direct Clustering Algorithm")
    print("="*80)
    
    matrix_step1 = dca_step1_initial_ordering(matrix)
    print("\nMatrix after Step 1:")
    print(matrix_step1)
    print("\n" + "="*80)
    
    matrix_step2 = dca_step2_shift_columns_left(matrix_step1)
    print("\nMatrix after Step 2:")
    print(matrix_step2)
    print("\n" + "="*80)
    
    matrix_step3 = dca_step3_shift_rows_up(matrix_step2)
    print("\nMatrix after Step 3 (FINAL):")
    print(matrix_step3)
    print("\n" + "="*80)
    
    return matrix_step3

In [96]:
# implement
dca_result = dca_algorithm(incidence_df)

DCA ALGORITHM - Direct Clustering Algorithm

Matrix after Step 1:
     K  A  L  M  B  C  D  G  F  H  E  J  I
P6   0  1  0  0  1  1  1  0  0  1  0  1  1
P1   0  1  0  0  1  1  1  0  0  0  0  1  1
P7   0  0  0  0  0  1  1  0  1  0  1  1  1
P2   0  1  0  0  0  1  1  0  0  1  0  1  0
P3   0  0  0  0  1  1  1  0  0  0  0  1  1
P4   0  1  0  0  1  0  1  1  0  1  0  0  0
P9   0  0  0  0  0  0  0  1  1  0  1  1  1
P12  0  0  0  0  0  0  0  1  1  0  1  1  1
P13  0  0  0  0  0  0  0  1  1  1  1  0  1
P15  0  0  0  0  0  0  0  1  1  1  1  1  0
P5   0  0  0  0  1  1  1  0  0  0  0  0  1
P8   0  0  0  0  0  0  0  0  0  1  1  1  1
P10  0  0  0  0  0  0  0  0  1  0  1  1  1
P14  0  0  0  0  0  0  0  1  1  1  1  0  0
P16  0  0  0  0  0  0  0  0  1  1  0  1  1
P11  0  0  0  0  0  0  0  1  0  0  1  0  1
P17  1  0  1  1  0  0  0  0  0  0  0  0  0
P18  1  0  1  1  0  0  0  0  0  0  0  0  0
P20  1  0  1  1  0  0  0  0  0  0  0  0  0
P19  0  0  1  1  0  0  0  0  0  0  0  0  0


Matrix after Step 2:
     A  

### Note: Use Steps rather than using Machines for the matrix

because:


To start with simplification, the __multi functional machines are not considered here__, which makes the capacity planning of them to be none. It is said to use steps as axis 1, which is rather use unifunctional machines as axis 1. This aligns with DCA purpooses

In [97]:
# Save the DCA reordered matrix
dca_result.to_csv('dca_reordered_matrix.csv')

in current repo, the raw output file is handled in excel for better visualization and other analysis and be saved as .xlsx file, feel free to download and check 

### Calculate From-to Chart

In [98]:
# load demand data and calculate for part demand

products = ['A1', 'A2', 'A3', 'B1', 'B2']

# weekly demand for products
demand_data = {
    'Weekly_Mean': [
        962,  
        1923, 
        2500, 
        1154, 
        1538  
    ],
    'Weekly_CV': [0.15, 0.2, 0.2, 0.12, 0.18] 
}

demand_df = pd.DataFrame(demand_data, index=products) 

# the BOM of requirements of each Products (A, B) from each Parts (P1~P20)
bom_df = pd.read_csv('../Data/BOM.csv', index_col=0).T # for the later matrix multiplication dim align

# calculate the target capacity for products
Z_score = norm.ppf(0.995)

demand_df['Weekly_std'] = demand_df['Weekly_Mean'] * demand_df['Weekly_CV']
# round up to maintain integer
demand_df['Product_Capacity'] = np.ceil(demand_df['Weekly_Mean'] + demand_df['Weekly_std']*Z_score)

# Aggregate to parts demand
part_demand = pd.DataFrame()
part_demand['Part_Weekly_Capacity'] = bom_df.dot(demand_df['Product_Capacity'])

part_demand['Part_Weekly_Capacity'] = np.ceil(part_demand['Part_Weekly_Capacity']) 

# Fix: Strip whitespace from the index
part_demand.index = part_demand.index.str.strip()

# Smooth the weekly demand into hourly demand
hr_demand = part_demand / (5*16)
hr_demand = np.ceil(hr_demand)
hr_demand.rename(columns={"Part_Weekly_Capacity": "Hourly Demand"}, inplace=True)

print('\n hourly capacity for each part:\n', hr_demand)

print(hr_demand.columns)


 hourly capacity for each part:
      Hourly Demand
P1           383.0
P2           200.0
P3           104.0
P4           169.0
P5           132.0
P6            73.0
P7           259.0
P8            76.0
P9           168.0
P10          229.0
P11          146.0
P12          180.0
P13           90.0
P14          335.0
P15           38.0
P16          226.0
P17           90.0
P18          216.0
P19          366.0
P20          272.0
Index(['Hourly Demand'], dtype='object')


In [99]:
# define product families from DCA results -- can check the excel file for better visualization

family_structure = {
    'D1': {
        'products': ['P1', 'P2', 'P3', 'P4', 'P5', 'P6'],
        'machines': ['A', 'B', 'C', 'D', 'ABC', 'CD']
    },
    'D2': {
        'products': ['P7', 'P8', 'P9', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16'],
        'machines': ['H', 'I', 'J']
    },
    'D3': {
        'products': ['P7', 'P8', 'P9', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16'],
        'machines': ['E', 'F', 'G', 'EG']
    },
    'D4': {
        'products': ['P17', 'P18', 'P19', 'P20'],
        'machines': ['K', 'L', 'M', 'KLM']
    }
}

In [100]:
rt = steps_df.fillna("")
pt = pd. read_csv("../Data/Timecost.csv").fillna(0)
with open("Allocation.json", "r") as f:
    allocation = json.load(f)

In [101]:
def build_ft_for_family(fam_name: str, fam_info: dict, allocation_data: dict) -> pd.DataFrame:

    fam_products = fam_info['products']
    fam_nodes = fam_info['machines']

    ft = defaultdict(lambda: defaultdict(int))

    for p in fam_products:
        seq   = list(s for s in rt.loc[rt['Part']==p].iloc[0, 1:] if str(s).strip() != "")
        times = list(float(t) for t in pt.loc[pt['Part']==p].iloc[0, 1:] if pd.notna(t)) # minutes/unit (0 means no time)

        for i in range(len(seq) - 1):
            step_from, step_to = seq[i], seq[i+1]
            t = times[i]

            # Find ALL machines that handle this product-step combination for FROM step
            machines_from = []
            for machine_type, machine_data in allocation_data.items():
                for alloc in machine_data["allocations"]:
                    if alloc["product"] == p and alloc["step"] == step_from:
                        if machine_type in fam_nodes:
                            machines_from.append({
                                "machine": machine_type,
                                "hours": alloc["hours"]
                            })

            # Find ALL machines that handle this product-step combination for TO step
            machines_to = []
            for machine_type, machine_data in allocation_data.items():
                for alloc in machine_data["allocations"]:
                    if alloc["product"] == p and alloc["step"] == step_to:
                        if machine_type in fam_nodes:
                            machines_to.append(machine_type)

            # Skip if no machines found or zero time
            if len(machines_from) == 0 or len(machines_to) == 0 or t == 0:
                continue

            # Calculate flow for each FROM machine to each TO machine
            for mfrom in machines_from:
                for mto in machines_to:
                    daily = mfrom["hours"] * 60 / (t * 5)
                    flow = np.ceil(daily)
                    if flow > 0:
                        ft[mfrom["machine"]][mto] += flow

    # square DF (zeros by default)
    df = pd.DataFrame(0, index=fam_nodes, columns=fam_nodes, dtype=int)
    for i in ft:
        for j, v in ft[i].items():
            if i in df.index and j in df.columns:
                df.loc[i, j] = v
    return df


outputs = {}
for fam in ("D1", "D2", "D3", "D4"):
    df = build_ft_for_family(fam, family_structure[fam], allocation)
    outputs[fam] = df
    print(f"\nfrom-to-chart for family {fam} (Daily):\n", df)


from-to-chart for family D1 (Daily):
         A     B     C     D   ABC   CD
A       0  2465   797     0   676  797
B    1531     0  2056   950  3355    0
C       0     0     0  1576     0   80
D       0     0     0     0     0  415
ABC     0    35     0   995     0    0
CD      0     0     0   718     0  718

from-to-chart for family D2 (Daily):
    H     I     J
H  0  1263   455
I  0     0  4242
J  0   303     0

from-to-chart for family D3 (Daily):
       E     F     G    EG
E     0  3646  1991  1195
F     0     0  2817   671
G   996   511     0     0
EG  259   718     0   194

from-to-chart for family D4 (Daily):
         K     L     M  KLM
K       0  1172  2382  863
L    1950     0  3470  359
M       0  1464     0    0
KLM     1    51     0    0


### Generate Flow Proportion Data for Layout Planning

For each part, we need:
- Daily demand
- Step sequence with machines and proportions

This intends to serves as instructions to route the flow bewtween multi functional machines and single usage ones, but due to time limit, this as a part of numerical analysis for more detailed planning of flows, leave asis

In [102]:
def generate_flow_proportions(steps_df, allocation_data, hr_demand):
    """
    Generate flow proportion data structure for all parts.
    
    For each part:
    - Calculate daily demand (hourly * 16 hours/day)
    - Extract step sequence
    - For each step, find which machines handle it and their proportions
    
    Returns:
        dict: Nested dictionary with structure:
              {part: {daily_demand: float, step_1: {step_name: str, machines: {machine: proportion}}}}
    """
    flow_data = {}
    
    for idx, row in steps_df.iterrows():
        part = row['Part']
        
        # Calculate daily demand
        hourly_demand = hr_demand.loc[part, 'Hourly Demand']
        daily_demand = hourly_demand * 16  # 16 hours per day
        
        # Initialize part data
        part_data = {
            'daily_demand': float(daily_demand)
        }
        
        # Extract step sequence
        step_sequence = []
        for col in steps_df.columns[1:]:  # Skip 'Part' column
            step = row[col]
            if pd.notna(step) and step.strip() != '':
                step_sequence.append(step.strip())
        
        # For each step, find machines and calculate proportions
        for step_idx, step_name in enumerate(step_sequence, start=1):
            # Find all machines that handle this (part, step) combination
            machine_hours = {}
            
            for machine_type, machine_data in allocation_data.items():
                for alloc in machine_data['allocations']:
                    if alloc['product'] == part and alloc['step'] == step_name:
                        machine_hours[machine_type] = alloc['hours']
            
            # Calculate proportions
            total_hours = sum(machine_hours.values())
            
            if total_hours > 0:
                machine_proportions = {
                    machine: hours / total_hours 
                    for machine, hours in machine_hours.items()
                }
            else:
                machine_proportions = {}
            
            # Add to part data
            part_data[f'step_{step_idx}'] = {
                'step_name': step_name,
                'machines': machine_proportions
            }
        
        flow_data[part] = part_data
    
    return flow_data


# Generate the flow proportion data
flow_proportions = generate_flow_proportions(steps_df, allocation, hr_demand)

# Save to JSON file
with open('flow_proportions.json', 'w') as f:
    json.dump(flow_proportions, f, indent=4)

print("Flow proportion data generated and saved to 'flow_proportions.json'")
print(f"\nGenerated data for {len(flow_proportions)} parts")
print("\nSample output for P1:")
print(json.dumps(flow_proportions['P1'], indent=2))

Flow proportion data generated and saved to 'flow_proportions.json'

Generated data for 20 parts

Sample output for P1:
{
  "daily_demand": 6128.0,
  "step_1": {
    "step_name": "B",
    "machines": {
      "B": 1.0
    }
  },
  "step_2": {
    "step_name": "A",
    "machines": {
      "A": 0.9775774206193649,
      "ABC": 0.022422579380635046
    }
  },
  "step_3": {
    "step_name": "B",
    "machines": {
      "B": 1.0
    }
  },
  "step_4": {
    "step_name": "C",
    "machines": {
      "ABC": 0.36590827126617015,
      "C": 0.6340917287338299
    }
  },
  "step_5": {
    "step_name": "D",
    "machines": {
      "D": 1.0
    }
  },
  "step_6": {
    "step_name": "I",
    "machines": {
      "I": 1.0
    }
  },
  "step_7": {
    "step_name": "J",
    "machines": {
      "J": 1.0
    }
  }
}


## Cost Analysis

Calculate material handling costs based on:
- Complete from-to-chart (daily flows across ALL machines)
- Rectilinear distances between departments
- Pallet jack speed: 264 ft/min
- Pallet capacity: 100 units per load
- Handler wage: $40/hr
- Pallets need to be returned (double the distance)
- Operating hours: 16 hr/day, 5 days/week, 49 weeks/year

In [103]:
# exact same iteration logic as to make from-to for each family
def build_complete_from_to_chart(allocation_data):
    """
    Build complete from-to chart across ALL machines using production allocation logic.
    Uses the exact same logic as build_ft_for_family but without family restrictions.
    
    Parameters:
    - allocation_data: dict from Allocation.json showing machine allocations
    
    Returns:
    - DataFrame with daily flows between all machine types
    """
    
    # Get all machine types from allocation data
    all_machines = sorted(allocation_data.keys())
    
    # Initialize from-to dictionary
    ft = defaultdict(lambda: defaultdict(int))
    
    # Get all unique products
    all_products = set()
    for machine_data in allocation_data.values():
        for alloc in machine_data["allocations"]:
            all_products.add(alloc["product"])
    
    # Process each product
    for p in all_products:
        # Get step sequence for this product
        seq = list(s for s in rt.loc[rt['Part']==p].iloc[0, 1:] if str(s).strip() != "")
        times = list(float(t) for t in pt.loc[pt['Part']==p].iloc[0, 1:] if pd.notna(t))
        
        # Process consecutive step pairs
        for i in range(len(seq) - 1):
            step_from, step_to = seq[i], seq[i+1]
            t = times[i]
            
            # Find ALL machines that handle this product-step combination for FROM step
            machines_from = []
            for machine_type, machine_data in allocation_data.items():
                for alloc in machine_data["allocations"]:
                    if alloc["product"] == p and alloc["step"] == step_from:
                        machines_from.append({
                            "machine": machine_type,
                            "hours": alloc["hours"]
                        })
            
            # Find ALL machines that handle this product-step combination for TO step
            machines_to = []
            for machine_type, machine_data in allocation_data.items():
                for alloc in machine_data["allocations"]:
                    if alloc["product"] == p and alloc["step"] == step_to:
                        machines_to.append(machine_type)
            
            # Skip if no machines found or zero time
            if len(machines_from) == 0 or len(machines_to) == 0 or t == 0:
                continue
            
            # Calculate flow for each FROM machine to each TO machine
            for mfrom in machines_from:
                for mto in machines_to:
                    # Daily flow = (hours allocated * 60 min/hr) / (time per unit * 5 days/week)
                    daily = mfrom["hours"] * 60 / (t * 5)
                    flow = np.ceil(daily)
                    if flow > 0:
                        ft[mfrom["machine"]][mto] += flow
    
    # Convert to square DataFrame
    df = pd.DataFrame(0, index=all_machines, columns=all_machines, dtype=int)
    for i in ft:
        for j, v in ft[i].items():
            if i in df.index and j in df.columns:
                df.loc[i, j] = v
    
    return df


# Build the complete from-to chart
print("="*80)
print("BUILDING COMPLETE FROM-TO CHART (ALL MACHINES)")
print("="*80)

complete_from_to = build_complete_from_to_chart(allocation)

print("\nComplete From-To Chart (Daily flows):")
print(complete_from_to)
print("\n")

# Show summary statistics
total_daily_flow = complete_from_to.sum().sum()
non_zero_routes = (complete_from_to > 0).sum().sum()
print(f"Total daily flow across all routes: {total_daily_flow:,} units")
print(f"Number of non-zero routes: {non_zero_routes}")

# Save to CSV
complete_from_to.to_csv('complete_from_to_chart.csv')
print(f"\nSaved to 'complete_from_to_chart.csv'")
print("="*80)

BUILDING COMPLETE FROM-TO CHART (ALL MACHINES)

Complete From-To Chart (Daily flows):
        A   ABC     B     C   CD     D    E    EG     F     G     H     I  \
A       0   676  2465   797  797     0    0     0     0     0     0     0   
ABC     0     0    35     0    0   995    0     0     0     0     0     0   
B    1531  3355     0  2056    0   950    0     0     0     0     0     0   
C       0     0     0     0   80  2610    0     0     0     0     0     0   
CD      0     0     0     0  718   718    0     0     0     0    71   415   
D       0     0     0     0  415     0    0     0     0   676  1018  3090   
E       0     0     0     0    0     0    0  1195  3646  1991   303     0   
EG      0     0     0     0    0     0  259   194   718     0     0   388   
F       0     0     0  1034    0     0    0   671     0  2817  1056  1633   
G       0     0     0     0    0     0  996     0   511     0  2732  1786   
H       0     0     0     0    0     0    0     0     0     0     0

In [104]:
# Coordination taken from CAD model
coordinates = {
    'A': (24.5, 64.9),
    'B': (48.33, 78.1),
    'C': (35.5, 86.8),
    'D': (50.5, 86.8),
    'E': (52, 159.5),
    'F': (101.73, 145.36),
    'G': (130, 167),
    'H': (169.6, 91),
    'I': (206, 91),
    'L': (70.8, 51),
    'M': (103.5, 51),
    'K': (86, 65),
    'J': (251.2, 77.7),
    'ABC': (50.5, 86.8),
    'CD': (125.5, 104.5),
    'EG': (67, 129),
    'KLM': (83.5, 37)
}

# Parameters
PALLET_CAPACITY = 100  # units per pallet
PALLET_JACK_SPEED = 264  # ft/min
HANDLER_WAGE = 40  # $/hr
HOURS_PER_DAY = 16
WEEKS_PER_YEAR = 49
DAYS_PER_WEEK = 5

In [105]:
def rectilinear_distance(coord1, coord2):
    """Calculate Manhattan/rectilinear distance between two coordinates"""
    return abs(coord1[0] - coord2[0]) + abs(coord1[1] - coord2[1])

def build_distance_matrix(coords):
    """Build distance matrix for all locations"""
    locations = list(coords.keys())
    n = len(locations)
    
    dist_matrix = pd.DataFrame(0.0, index=locations, columns=locations)
    
    for i in locations:
        for j in locations:
            dist_matrix.loc[i, j] = rectilinear_distance(coords[i], coords[j])
    
    return dist_matrix

# Calculate distance matrix
distance_matrix = build_distance_matrix(coordinates)
print("Distance Matrix (rectilinear distance in feet):")
print(distance_matrix)

Distance Matrix (rectilinear distance in feet):
          A       B       C       D       E       F       G       H       I  \
A      0.00   37.03   32.90   47.90  122.10  157.69  207.60  171.20  207.60   
B     37.03    0.00   21.53   10.87   85.07  120.66  170.57  134.17  170.57   
C     32.90   21.53    0.00   15.00   89.20  124.79  174.70  138.30  174.70   
D     47.90   10.87   15.00    0.00   74.20  109.79  159.70  123.30  159.70   
E    122.10   85.07   89.20   74.20    0.00   63.87   85.50  186.10  222.50   
F    157.69  120.66  124.79  109.79   63.87    0.00   49.91  122.23  158.63   
G    207.60  170.57  174.70  159.70   85.50   49.91    0.00  115.60  152.00   
H    171.20  134.17  138.30  123.30  186.10  122.23  115.60    0.00   36.40   
I    207.60  170.57  174.70  159.70  222.50  158.63  152.00   36.40    0.00   
L     60.20   49.57   71.10   56.10  127.30  125.29  175.20  138.80  175.20   
M     92.90   82.27  103.80   88.80  160.00   96.13  142.50  106.10  142.50   
K   

In [106]:
# Calculate travel cost using the complete from-to matrix
def calculate_complete_travel_cost(from_to_matrix, distance_matrix):
    """
    Calculate annual travel cost for material handling across all machines each year
    
    Parameters:
    - from_to_matrix: DataFrame with daily flows between all machines
    - distance_matrix: DataFrame with rectilinear distances
    
    Returns:
    - total_cost: Annual cost in dollars
    - cost_breakdown: DataFrame with detailed calculations
    """
    
    # Initialize results storage
    results = []
    
    # Iterate through all from-to combinations
    for from_machine in from_to_matrix.index:
        for to_machine in from_to_matrix.columns:
            daily_flow = from_to_matrix.loc[from_machine, to_machine]
            
            # Skip if no flow or same machine
            if daily_flow == 0 or from_machine == to_machine:
                continue
            
            # Check if both machines have coordinates
            if from_machine not in distance_matrix.index or to_machine not in distance_matrix.columns:
                print(f"Warning: Missing coordinates for {from_machine} -> {to_machine}")
                continue
            
            # Get distance
            distance = distance_matrix.loc[from_machine, to_machine]
            
            # Calculate number of trips per day
            # Each trip carries PALLET_CAPACITY units
            trips_per_day = np.ceil(daily_flow / PALLET_CAPACITY)
            
            # Calculate round-trip distance (pallets must be returned)
            round_trip_distance = 2 * distance
            
            # Calculate time per trip (in minutes)
            time_per_trip_min = round_trip_distance / PALLET_JACK_SPEED
            
            # Calculate daily time (in hours)
            daily_time_hrs = (trips_per_day * time_per_trip_min) / 60
            
            # Calculate annual time (in hours)
            annual_time_hrs = daily_time_hrs * DAYS_PER_WEEK * WEEKS_PER_YEAR
            
            # Calculate annual cost
            annual_cost = annual_time_hrs * HANDLER_WAGE
            
            # Store results
            results.append({
                'From': from_machine,
                'To': to_machine,
                'Daily_Flow_units': daily_flow,
                'Distance_ft': distance,
                'Round_Trip_ft': round_trip_distance,
                'Trips_Per_Day': trips_per_day,
                'Time_Per_Trip_min': time_per_trip_min,
                'Daily_Time_hrs': daily_time_hrs,
                'Annual_Time_hrs': annual_time_hrs,
                'Annual_Cost_$': annual_cost
            })
    
    # Create breakdown DataFrame
    breakdown_df = pd.DataFrame(results)
    
    # Sort by annual cost (descending) to see most expensive routes first
    breakdown_df = breakdown_df.sort_values('Annual_Cost_$', ascending=False)
    
    # Calculate total cost
    total_cost = breakdown_df['Annual_Cost_$'].sum()
    total_annual_hours = breakdown_df['Annual_Time_hrs'].sum()
    
    print(f"\n{'='*80}")
    print(f"COMPLETE TRAVEL COST ANALYSIS - ALL MACHINES")
    print(f"{'='*80}")
    print(f"\nTotal Annual Handler Hours: {total_annual_hours:,.2f} hours")
    print(f"Total Annual Handler Cost: ${total_cost:,.2f}")
    print(f"\nTop 20 Most Expensive Routes:")
    print(breakdown_df.head(20).to_string(index=False))
    print(f"\n{'='*80}")
    print(f"\nFull breakdown with all {len(breakdown_df)} routes:")
    print(breakdown_df.to_string(index=False))
    
    return total_cost, breakdown_df


# Calculate the complete travel cost
total_annual_cost, cost_breakdown = calculate_complete_travel_cost(complete_from_to, distance_matrix)

# Save the detailed breakdown
cost_breakdown.to_csv('travel_cost_breakdown.csv', index=False)
print(f"\n\nDetailed cost breakdown saved to 'travel_cost_breakdown.csv'")

# Print summary statistics
print(f"\n{'='*80}")
print(f"SUMMARY STATISTICS")
print(f"{'='*80}")
print(f"Total number of routes with flow: {len(cost_breakdown)}")
print(f"Total annual handler cost: ${total_annual_cost:,.2f}")
print(f"Total annual handler hours: {cost_breakdown['Annual_Time_hrs'].sum():,.2f} hours")
print(f"Average cost per route: ${cost_breakdown['Annual_Cost_$'].mean():,.2f}")
print(f"Median cost per route: ${cost_breakdown['Annual_Cost_$'].median():,.2f}")
print(f"Most expensive route: {cost_breakdown.iloc[0]['From']} -> {cost_breakdown.iloc[0]['To']} (${cost_breakdown.iloc[0]['Annual_Cost_$']:,.2f})")
print(f"{'='*80}")


COMPLETE TRAVEL COST ANALYSIS - ALL MACHINES

Total Annual Handler Hours: 1,374.49 hours
Total Annual Handler Cost: $54,979.77

Top 20 Most Expensive Routes:
From To  Daily_Flow_units  Distance_ft  Round_Trip_ft  Trips_Per_Day  Time_Per_Trip_min  Daily_Time_hrs  Annual_Time_hrs  Annual_Cost_$
   D  I              3090       159.70         319.40           31.0           1.209848        0.625088       153.146654    6125.866162
   I  J              6481        58.50         117.00           65.0           0.443182        0.480114       117.627841    4705.113636
   G  H              2732       115.60         231.20           28.0           0.875758        0.408687       100.128283    4005.131313
   G  I              1786       152.00         304.00           18.0           1.151515        0.345455        84.636364    3385.454545
   F  I              1633       158.63         317.26           17.0           1.201742        0.340494        83.420953    3336.838131
   E  F              3646

In [111]:
# overall cost for the whole facility
overall = total_annual_cost * 4
print(f"the annual handling cost for the whole facility:", '$', round(overall, 3))

the annual handling cost for the whole facility: $ 219919.078
